In [ ]:
!pip install -q langchain langchain-google-genai langchain-community faiss-cpu pypdf

import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('Ragproject')

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("Set- Up- Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 15.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth

/tmp/ipykernel_580/1875035598.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
from google.colab import files
upload = files.upload()

pdf_name=list(upload.keys())[0]
print(pdf_name)

Saving employee_handbook.pdf to employee_handbook.pdf
employee_handbook.pdf


In [ ]:
docs = PyPDFLoader(pdf_name).load()

print("Total number of pages: ", len(docs))
print("Total number of characters: ", sum(len(d.page_content) for d in docs))
print()
print(docs[0].page_content[:500])

Total number of pages:  3
Total number of characters:  5208

Employee Handbook
 Practice dataset for LangChain RAG Project 2 — AI Employee Handbook Assistant
Note: This is a fictional practice document created for learning purposes. It is not an actual company's
employee policy.
1. Working Hours
 The standard working hours are Monday to Friday, from 9:00 AM to 6:00 PM.
 Employees receive a one-hour lunch break between 1:00 PM and 2:00 PM.
 Employees are expected to be available during working hours unless they are on approved leave or have
an approved 


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)

chunks = splitter.split_documents(docs)
print("Total number of chunks: ",len(chunks))

print()
print(chunks[0].page_content[-150:])
print()
print(chunks[1].page_content[:150])

Total number of chunks:  9

endance system each working day.
 An employee who cannot attend work should inform the reporting manager before 9:30 AM whenever
reasonably possible.

 An employee who cannot attend work should inform the reporting manager before 9:30 AM whenever
reasonably possible.
 Repeated unexplained absences 


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks,embeddings)

print("Totally",len(chunks)," chunks stored in vector store")

Totally 9  chunks stored in vector store


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

found = retriever.invoke("How do I get my fees back if I cancel admission?")

for d in found:
    print(d.page_content[:200])
    print("-" * 60)

 Employees who complete six months of continuous service are eligible for the company's learning and
development reimbursement program.
 Benefit eligibility may vary according to employment status a
------------------------------------------------------------
 Before the final working day, employees must return company property and complete the required
knowledge-transfer and exit procedures.
14. Remote Work Rules
 Employees working remotely must maintai
------------------------------------------------------------
review.
12. Notice Period
 Confirmed employees are normally required to serve a 60-day notice period when resigning.
 Employees on probation are normally required to serve a 30-day notice period.
 
------------------------------------------------------------
 Unused casual leave cannot be converted into cash at the end of the calendar year.
7. Public Holidays
 The company observes 12 public holidays each calendar year.
 The holiday calendar is publishe
-------------

In [ ]:
for d, score in vectorstore.similarity_search_with_score("How do I get my fees back if I cancel admission?", k=3):
    print(f"distance: {score:.3f}  ->  {d.page_content[:80]}")

distance: 0.880  ->   Employees who complete six months of continuous service are eligible for the c
distance: 0.925  ->   Before the final working day, employees must return company property and compl
distance: 0.932  ->  review.
12. Notice Period
 Confirmed employees are normally required to serve a


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You answer questions using only the context provided.
     If the answer is not available in the context, reply exactly:
     'I could not find this in the document.'
     Do not guess or use outside knowledge.
     Give a complete answer using all information from the context
     that is directly relevant to the question.
     If the context contains conditions, exceptions, requirements,
     time periods, approvals, or related details that affect the
     answer, include them.
     Do not include unrelated information."""
    ),
    ("human",
     "CONTEXT FROM THE DOCUMENT:\n\n{context}\n\nQUESTION: {question}")
])

def format_docs(docs):
    return "\n\n-----------\n\n".join(d.page_content for d in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Chain ready.")

Chain ready.


In [ ]:
print(rag_chain.invoke("what is the fee refund policy for canceled admission"))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


I could not find this in the document.


In [ ]:
print(rag_chain.invoke("who is Nachammai"))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


I could not find this in the document.


In [ ]:
print(rag_chain.invoke("What is the notice period for an employee who is on probatio"))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Employees on probation are normally required to serve a 30-day notice period.
